# Piper TTS 한국어 훈련 (Google Colab T4)

**런타임 설정**: 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 실행

| 항목 | 내용 |
|------|------|
| GPU | T4 (16GB VRAM, 무료) |
| 세션 시간 | 최대 12시간 (주기적 저장 필수) |
| 체크포인트 | Google Drive 자동 저장 |
| 테스트 데이터 | KSS 공개 데이터셋 (12시간, 12,853 문장) |

---
**⚠️ 세션이 끊겨도 Drive에 저장된 체크포인트에서 재개됩니다.**

## ⚠️ Step 0: Python 3.11로 전환 (최초 1회)

`piper-phonemize`의 Linux 바이너리 휠은 **Python 3.10 / 3.11**까지만 제공됩니다.  
Colab이 Python 3.12인 경우 아래 셀을 **먼저** 실행하세요.  
런타임이 자동 재시작되면 Step 1부터 다시 실행하면 됩니다.

In [ ]:
import sys

py_ver = f"{sys.version_info.major}.{sys.version_info.minor}"
print(f"현재 Python: {py_ver}")

if sys.version_info >= (3, 12):
    print("Python 3.12+ 감지 → condacolab로 3.11 전환 중...")
    import subprocess
    subprocess.run(["pip", "install", "condacolab", "-q"], check=True)
    import condacolab
    condacolab.install()  # 런타임 자동 재시작됨
else:
    print(f"✓ Python {py_ver} — 전환 불필요. Step 1부터 진행하세요.")

## Step 1: GPU 및 환경 확인

In [ ]:
import torch
import subprocess

print(f'PyTorch: {torch.__version__}')
print(f'CUDA 사용 가능: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️ GPU 없음 - 런타임 유형을 T4 GPU로 변경하세요')

result = subprocess.run(['df', '-h', '/'], capture_output=True, text=True)
print('\n디스크:')
print(result.stdout)

## Step 2: Google Drive 연결 (체크포인트 저장용)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/piper-korean'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
print(f'체크포인트 저장 경로: {DRIVE_DIR}/checkpoints')

## Step 3: 시스템 의존성 설치

In [ ]:
%%bash
apt-get update -qq
apt-get install -y -qq \
    espeak-ng \
    libespeak-ng-dev \
    build-essential \
    cython3 \
    libsndfile1

# 한국어 음소 변환 테스트
echo -n '한국어 espeak-ng 테스트: '
espeak-ng -v ko -q --ipa '안녕하세요' && echo '✓ OK'

## Step 4: Piper-train 설치

In [ ]:
import subprocess, sys, os

def pip(*args):
    """커널과 동일한 Python으로 pip 실행"""
    cmd = [sys.executable, '-m', 'pip'] + list(args)
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise RuntimeError(f"pip 실패: {' '.join(args[:3])}")

def shell(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout: print(r.stdout.strip())
    if r.stderr: print(r.stderr[-500:].strip())
    return r.returncode

print(f"Python: {sys.executable}  ({sys.version.split()[0]})")

# ── pip 23.3.2 고정 ──────────────────────────────────────────
pip('install', '-q', 'pip==23.3.2')
print("✓ pip 23.3.2")

# ── piper 소스 클론 ───────────────────────────────────────────
if not os.path.exists('/content/piper'):
    shell('git clone --depth 1 https://github.com/rhasspy/piper.git /content/piper')
print("✓ piper 소스")

# ── piper-phonemize ──────────────────────────────────────────
minor = sys.version_info.minor
installed = False
for ver in [f'cp3{minor}', 'cp311', 'cp310']:
    url = (f'https://github.com/rhasspy/piper-phonemize/releases/download/v1.1.0/'
           f'piper_phonemize-1.1.0-{ver}-{ver}-manylinux_2_28_x86_64.whl')
    try:
        pip('install', '-q', url)
        installed = True
        print(f"✓ piper-phonemize ({ver})")
        break
    except RuntimeError:
        continue
if not installed:
    raise RuntimeError("piper-phonemize 설치 실패 — Step 0 (Python 3.11 전환)을 먼저 실행하세요")

# ── pytorch-lightning 1.7.7 ──────────────────────────────────
pip('install', '-q', 'pytorch-lightning==1.7.7')
print("✓ pytorch-lightning 1.7.7")

# ── piper-train ──────────────────────────────────────────────
pip('install', '-q', '-e', '/content/piper/src/python')
print("✓ piper-train 설치")

# ── monotonic_align Cython 빌드 ──────────────────────────────
rc = shell('cd /content/piper/src/python && bash build_monotonic_align.sh')
print("✓ monotonic_align 빌드" if rc == 0 else "⚠ monotonic_align 빌드 실패 (훈련은 가능)")

print("\n✓ 모든 설치 완료")

In [ ]:
import sys, subprocess

# editable install 경로가 누락된 경우 대비해 직접 추가
PIPER_SRC = '/content/piper/src/python'
if PIPER_SRC not in sys.path:
    sys.path.insert(0, PIPER_SRC)

# 그래도 import 안 되면 재설치
try:
    import piper_train
except ModuleNotFoundError:
    print("piper_train 경로 없음 → 재설치 중...")
    subprocess.run(
        ['pip', 'install', '-q', '-e', PIPER_SRC],
        check=True
    )
    import piper_train

import piper_phonemize
import pytorch_lightning as pl

result = piper_phonemize.phonemize_espeak('안녕하세요 반갑습니다', 'ko')
print(f'piper_phonemize 한국어 테스트: {result}')
print(f'pytorch_lightning: {pl.__version__}')
print(f'piper_train 경로: {piper_train.__file__}')
print('✓ 모든 패키지 정상')

## Step 5: 학습 데이터 준비

### 옵션 A: KSS 공개 데이터셋 (테스트용, 권장)
- 한국어 단일 화자 (여성), 12,853 문장, 약 12시간
- 공개 라이선스 (CC BY 4.0)

### 옵션 B: 내 목소리 데이터 업로드
- LJSpeech 형식으로 준비 후 Drive에 업로드

In [ ]:
# === 옵션 A: KSS 데이터셋 다운로드 ===
# KSS는 AIHub 또는 Kaggle에서 다운로드 가능
# 여기서는 Kaggle API를 통한 방법 제시

USE_SAMPLE_DATA = True  # 빠른 테스트: True | 전체 KSS: False

import os, random

if USE_SAMPLE_DATA:
    # 테스트용 샘플 데이터 생성 (실제 KSS 없이도 파이프라인 확인)
    print('샘플 데이터 생성 중 (파이프라인 테스트용)...')
    
    import subprocess, struct, math
    
    os.makedirs('/content/recordings/wavs', exist_ok=True)
    
    SAMPLE_TEXTS = [
        '안녕하세요 반갑습니다',
        '오늘 날씨가 정말 좋네요',
        '한국어 음성 합성 테스트입니다',
        '파이퍼 TTS 훈련을 시작합니다',
        '인공지능 기술이 발전하고 있습니다',
        '목소리 데이터를 수집해야 합니다',
        '셰르파 온넥스로 음성을 합성합니다',
        '자연스러운 한국어 음성을 만들겠습니다',
        '딥러닝 모델을 학습시키는 중입니다',
        '좋은 결과가 나오기를 기대합니다',
    ]
    
    def make_sine_wav(path, freq=220, duration=2.0, sr=22050):
        """테스트용 더미 WAV 생성 (실제 훈련에는 사용 불가)"""
        n = int(sr * duration)
        samples = [int(32767 * math.sin(2 * math.pi * freq * i / sr)) for i in range(n)]
        with open(path, 'wb') as f:
            # WAV 헤더
            data_size = n * 2
            f.write(b'RIFF')
            f.write(struct.pack('<I', 36 + data_size))
            f.write(b'WAVEfmt ')
            f.write(struct.pack('<IHHIIHH', 16, 1, 1, sr, sr*2, 2, 16))
            f.write(b'data')
            f.write(struct.pack('<I', data_size))
            f.write(struct.pack(f'<{n}h', *samples))
    
    metadata_lines = []
    for i, text in enumerate(SAMPLE_TEXTS):
        fname = f'{i+1:04d}'
        make_sine_wav(f'/content/recordings/wavs/{fname}.wav',
                      freq=200 + i*10, duration=1.5)
        metadata_lines.append(f'{fname}|{text}|{text}')
    
    with open('/content/recordings/metadata.csv', 'w', encoding='utf-8') as f:
        f.write('\n'.join(metadata_lines))
    
    print(f'✓ 샘플 {len(SAMPLE_TEXTS)}개 생성완료')
    print('⚠️  이 데이터는 파이프라인 확인용입니다. 실제 훈련엔 진짜 음성 데이터가 필요합니다.')

else:
    # 옵션 B: Drive에 업로드한 데이터 사용
    DATA_PATH = f'{DRIVE_DIR}/recordings'  # Drive에 미리 올려두세요
    print(f'Drive 데이터 사용: {DATA_PATH}')
    os.symlink(DATA_PATH, '/content/recordings')

## Step 6: 전처리 (텍스트 → 음소 변환)

In [ ]:
%%bash
cd /content

python3 -m piper_train.preprocess \
    --language ko \
    --input-dir /content/recordings \
    --output-dir /content/dataset \
    --dataset-format ljspeech \
    --single-speaker \
    --sample-rate 22050

echo ''
echo '── 데이터셋 통계 ──'
wc -l /content/dataset/train.txt /content/dataset/valid.txt 2>/dev/null || true
ls -lh /content/dataset/

## Step 7: 훈련

In [ ]:
# 훈련 설정
import torch

BATCH_SIZE = 32          # T4 16GB: 32 권장. OOM 시 16으로 줄이기
MAX_EPOCHS = 10000       # 전체 훈련: 충분한 에폭 수
VALIDATION_SPLIT = 0.05  # 5% 검증 데이터
NUM_WORKERS = 2

# 기존 체크포인트 확인 (재개 훈련)
import os, glob
ckpt_files = glob.glob(f'{DRIVE_DIR}/checkpoints/*.ckpt')
RESUME_CKPT = max(ckpt_files, key=os.path.getmtime) if ckpt_files else None

if RESUME_CKPT:
    print(f'기존 체크포인트 발견: {RESUME_CKPT}')
    print('→ 이어서 훈련합니다')
else:
    print('체크포인트 없음 → 처음부터 훈련')

print(f'\n설정:')
print(f'  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'  batch_size: {BATCH_SIZE}')
print(f'  max_epochs: {MAX_EPOCHS}')
print(f'  체크포인트: {DRIVE_DIR}/checkpoints')

In [ ]:
import subprocess, shlex

cmd = [
    'python3', '-m', 'piper_train.train',
    '--dataset-dir', '/content/dataset',
    '--accelerator', 'gpu',
    '--devices', '1',
    '--batch-size', str(BATCH_SIZE),
    '--validation-split', str(VALIDATION_SPLIT),
    '--num-workers', str(NUM_WORKERS),
    '--max_epochs', str(MAX_EPOCHS),
    # Drive에 체크포인트 저장
    '--checkpoint-dir', f'{DRIVE_DIR}/checkpoints',
]

if RESUME_CKPT:
    cmd += ['--resume_from_checkpoint', RESUME_CKPT]

print('실행 명령:')
print(' '.join(cmd))
print('')

# 훈련 시작 (출력이 실시간으로 표시됩니다)
result = subprocess.run(cmd, cwd='/content')
print(f'\n훈련 종료 (exit code: {result.returncode})')

## Step 8: ONNX 내보내기

In [ ]:
import glob, os

# 최신 체크포인트 찾기
ckpt_files = glob.glob(f'{DRIVE_DIR}/checkpoints/*.ckpt')
if not ckpt_files:
    # 로컬 lightning_logs에서도 확인
    ckpt_files = glob.glob('/content/lightning_logs/**/checkpoints/*.ckpt', recursive=True)

if not ckpt_files:
    print('체크포인트 없음 - 훈련 먼저 실행하세요')
else:
    latest_ckpt = max(ckpt_files, key=os.path.getmtime)
    output_onnx = f'{DRIVE_DIR}/my-korean-voice.onnx'
    
    print(f'체크포인트: {latest_ckpt}')
    print(f'출력: {output_onnx}')
    
    !python3 -m piper_train.export_onnx \
        --checkpoint {latest_ckpt} \
        --output {output_onnx}
    
    import os
    if os.path.exists(output_onnx):
        size_mb = os.path.getsize(output_onnx) / 1e6
        print(f'\n✓ ONNX 내보내기 완료: {output_onnx} ({size_mb:.1f} MB)')
        print('→ 이 파일을 sherpa-onnx 서버의 MODEL_DIR에 넣으면 바로 사용 가능합니다')

## Step 9: 체크포인트 진행 상황 확인

훈련 중 별도 셀에서 실행하면 현재 상태를 볼 수 있습니다.

In [ ]:
import glob, os, time
from datetime import datetime

ckpt_files = glob.glob(f'{DRIVE_DIR}/checkpoints/*.ckpt')
local_ckpt = glob.glob('/content/lightning_logs/**/checkpoints/*.ckpt', recursive=True)
all_ckpts = ckpt_files + local_ckpt

if not all_ckpts:
    print('체크포인트 없음 (훈련 시작 전 또는 아직 저장 안 됨)')
else:
    print(f'체크포인트 목록 ({len(all_ckpts)}개):')
    for f in sorted(all_ckpts, key=os.path.getmtime):
        mtime = datetime.fromtimestamp(os.path.getmtime(f))
        size = os.path.getsize(f) / 1e6
        print(f'  {mtime.strftime("%H:%M:%S")} | {size:.0f}MB | {os.path.basename(f)}')

---
## 참고: 훈련 시간 가이드

| 환경 | 1시간 데이터 / 10,000 epoch | 비용 |
|------|---------------------------|------|
| Colab T4 (무료) | ~8~12시간 | 무료 (세션 제한 있음) |
| Colab A100 (Pro) | ~2~3시간 | 월 $12 |
| M4 Pro (CPU) | ~20~30시간 | 전기세만 |
| RunPod RTX 4090 | ~1~2시간 | $0.5~1 |

**팁:** 10,000 epoch 마다 체크포인트 품질을 들어보고, 충분하면 내보내기. 음질은 보통 3,000~5,000 epoch에서 이미 쓸 만해집니다.

## 내 목소리 데이터 준비 시

```
recordings/
  wavs/
    0001.wav   ← 22050Hz, mono, 16-bit, 5~10초 내외
    0002.wav
    ...
  metadata.csv  ← 파일명|텍스트|텍스트
```

`metadata.csv` 예시:
```
0001|안녕하세요 반갑습니다|안녕하세요 반갑습니다
0002|오늘 날씨가 정말 좋네요|오늘 날씨가 정말 좋네요
```

최소 **500문장** (30분), 권장 **1,000~3,000문장** (1~2시간)